# Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import pykitti
from pathlib import Path
import cv2

# Load KITTI
base = Path.home() / 'SensorTrust' / 'datasets' / 'kitti'
data = pykitti.raw(base_path=str(base), date='2011_09_26', drive='0009')

# Import proxies
from src.proxies.gps_proxy import extract_all_gps_proxies
from src.proxies.imu_proxy import extract_all_imu_proxies
from src.proxies.camera_proxy import extract_all_camera_proxies
from src.proxies.lidar_proxy import extract_all_lidar_proxies

print("Setup complete. All imports successful.")

## GPS Proxies

In [ ]:
# Extract GPS proxies
dt = 0.1035  # Your measured frame interval
gps = extract_all_gps_proxies(data.oxts, dt=dt)

# Plot
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8))

ax1.plot(gps['speed'])
ax1.set_ylabel('Speed (m/s)')
ax1.set_title('GPS Forward Speed')
ax1.grid(True, alpha=0.3)

ax2.plot(gps['heading'])
ax2.set_ylabel('Heading (rad)')
ax2.set_title('GPS Heading (bearing)')
ax2.grid(True, alpha=0.3)

ax3.plot(gps['heading_rate'])
ax3.set_ylabel('Heading Rate (rad/s)')
ax3.set_xlabel('Frame')
ax3.set_title('GPS Heading Rate')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Stats
print(f"Speed: mean={np.nanmean(gps['speed']):.2f}, max={np.nanmax(gps['speed']):.2f} m/s")
print(f"Speed NaN: {np.any(np.isnan(gps['speed']))}")
print(f"Heading NaN: {np.any(np.isnan(gps['heading']))} (first frame expected)")
print(f"Heading Rate NaN: {np.any(np.isnan(gps['heading_rate']))} (first 2 frames expected)")

## IMU Proxies

In [ ]:
# Extract IMU proxies (delta_v instead of absolute speed)
imu = extract_all_imu_proxies(data.oxts, dt=dt, window=5)
gps = extract_all_gps_proxies(data.oxts, dt=dt, window=5)

# Compare speed CHANGES, not absolute speeds
window = 5
gps_dv = gps['delta_v']
imu_dv = imu['delta_v']

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(gps_dv, label='GPS Δv (5-frame)')
ax1.plot(imu_dv, label='IMU Δv (5-frame)', alpha=0.7)
ax1.set_ylabel('Speed Change (m/s)')
ax1.set_title('Speed Change Over 5-Frame Window: GPS vs IMU')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(imu['yaw_rate'], label='IMU Yaw Rate')
ax2.plot(gps['heading_rate'], label='GPS Heading Rate', alpha=0.7)
ax2.set_ylabel('Rate (rad/s)')
ax2.set_xlabel('Frame')
ax2.set_title('IMU Yaw Rate vs GPS Heading Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Agreement on speed change
valid = ~(np.isnan(gps_dv) | np.isnan(imu_dv))
dv_diff = np.abs(gps_dv[valid] - imu_dv[valid])
print(f"Δv agreement (mean abs diff): {np.mean(dv_diff):.4f} m/s")
print(f"Δv agreement (max abs diff):  {np.max(dv_diff):.4f} m/s")

## CAmera Proxies

In [ ]:
# Load camera frames as numpy arrays
camera_frames = []
for i in range(min(50, len(data.cam2_files))):  # First 50 for speed
    cam_pil = data.get_cam2(i)
    camera_frames.append(np.array(cam_pil))

# Extract camera proxy
cam = extract_all_camera_proxies(camera_frames)

# Plot
plt.figure(figsize=(12, 4))
plt.plot(cam['flow_magnitude'])
plt.xlabel('Frame Pair')
plt.ylabel('Mean Optical Flow (px/frame)')
plt.title('Camera: Mean Optical Flow Magnitude (First 50 Frames)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Flow magnitude: mean={np.mean(cam['flow_magnitude']):.2f}, max={np.max(cam['flow_magnitude']):.2f} px/frame")

## Stationary Check

In [ ]:
# Stationary frames: 404-446
stationary_start = 404
stationary_end = 446

# GPS speed during stationary
stat_speed = gps['speed'][stationary_start:stationary_end]
print("=== STATIONARY CHECK (Frames 404-446) ===")
print(f"GPS speed: mean={np.mean(stat_speed):.4f} m/s, std={np.std(stat_speed):.4f} m/s")
print(f"GPS speed max deviation: {np.max(np.abs(stat_speed)):.4f} m/s")

## Heading Stability Check

In [ ]:
# Check GPS heading noise at low speed
stat_heading_rate = gps['heading_rate'][stationary_start:stationary_end]
valid_hr = stat_heading_rate[~np.isnan(stat_heading_rate)]

print("=== HEADING STABILITY AT STATIONARY ===")
print(f"Mean heading rate: {np.mean(valid_hr):.6f} rad/s")
print(f"Std heading rate:  {np.std(valid_hr):.6f} rad/s")
print(f"Max heading rate:  {np.max(np.abs(valid_hr)):.6f} rad/s")
print(f"\nHeading gate threshold candidate: {np.std(valid_hr) * 3:.6f} rad/s (3σ)")

## Optical Flow At Rest

In [ ]:
# Camera optical flow on stationary frames
print("=== OPTICAL FLOW AT REST ===")
print("Loading stationary camera frames...")

stat_cam_frames = []
for i in range(stationary_start, min(stationary_end, len(data.cam2_files))):
    stat_cam_frames.append(np.array(data.get_cam2(i)))

stat_cam = extract_all_camera_proxies(stat_cam_frames)

print(f"Stationary flow: mean={np.mean(stat_cam['flow_magnitude']):.4f} px/frame")
print(f"Stationary flow: std={np.std(stat_cam['flow_magnitude']):.4f} px/frame")
print(f"Stationary flow: max={np.max(stat_cam['flow_magnitude']):.4f} px/frame")

# Compare with moving flow (first 50 frames)
moving_flow_mean = np.mean(cam['flow_magnitude'])
print(f"\nMoving flow mean: {moving_flow_mean:.4f} px/frame")
print(f"Ratio moving/stationary: {moving_flow_mean / np.mean(stat_cam['flow_magnitude']):.2f}x")
print(f"Camera usable: {'YES' if moving_flow_mean / np.mean(stat_cam['flow_magnitude']) > 2.0 else 'MARGINAL'}")

In [ ]:
# ============================================================
# LiDAR Proxy Validation
# ============================================================

'''
LiDAR Proxy Validation: The ego-motion compensated ICP residual was evaluated on both moving and stationary segments. Moving segments produced a mean residual of 0.209 m, while stationary segments produced a lower mean residual of 0.158 m with very low variance (σ = 0.007 m). This behaviour is consistent with expectations, indicating successful ego-motion compensation and stable LiDAR proxy extraction. The proxy is therefore considered suitable for subsequent feature engineering and anomaly detection stages.
'''

'''
Purpose:
--------
Validate that the LiDAR motion proxy (ICP residual) behaves
reasonably during normal vehicle motion.

Method:
-------
1. Extract consecutive LiDAR scans.
2. Compensate for ego-motion using KITTI OXTS poses.
3. Compute ICP residual between neighbouring scans.
4. Analyze the residual statistics.

Interpretation:
---------------
A lower residual indicates that the scene structure remains
consistent after ego-motion compensation.

A stable residual curve without large spikes suggests:
    - Correct pose transformation
    - Stable LiDAR proxy extraction
    - Reasonable scene alignment

Expected Behaviour:
-------------------
Residuals should remain bounded and physically reasonable
(typically a few centimeters to tens of centimeters).
'''

print("\n=== LIDAR ICP RESIDUAL VALIDATION ===")

# Small subset first so validation doesn't take forever
velo_scans = [data.get_velo(i) for i in range(50)]
oxts_data = data.oxts[:50]

lidar = extract_all_lidar_proxies(
    velo_scans,
    oxts_data
)

residuals = lidar['icp_residual']

print(f"Mean residual: {np.mean(residuals):.4f} m")
print(f"Std residual:  {np.std(residuals):.4f} m")
print(f"Max residual:  {np.max(residuals):.4f} m")
print(f"Min residual:  {np.min(residuals):.4f} m")

plt.figure(figsize=(10, 4))
plt.plot(residuals)
plt.title("LiDAR ICP Residual")
plt.xlabel("Frame Pair")
plt.ylabel("Residual (m)")
plt.grid(True, alpha=0.3)
plt.show()

print(f"LiDAR usable: {'YES' if np.mean(residuals) > 0 else 'NO'}")

In [ ]:
# ============================================================
# LiDAR Stationary Validation
# ============================================================

'''
Purpose:
--------
Verify that the LiDAR proxy produces lower residuals when the
vehicle is stationary.

Why This Matters:
-----------------
When the vehicle is not moving, consecutive LiDAR scans should
observe nearly the same environment.

Therefore:
    Stationary Residual < Moving Residual

A significantly lower stationary residual indicates:
    - Successful ego-motion compensation
    - Low sensor noise
    - Stable proxy behaviour

Expected Outcome:
-----------------
Moving segments should show larger residuals because the scene
changes relative to the sensor.

Stationary segments should produce:
    - Lower mean residual
    - Lower variance
    - Smoother residual curve
'''

print("OXTS frames :", len(data.oxts))
print("LiDAR frames:", len(data.velo_files))

# Use the LAST 40 frames that exist in BOTH sensors
n = min(len(data.oxts), len(data.velo_files))

start_idx = max(0, n - 40)
end_idx = n

print(f"\nTesting stationary segment: {start_idx} -> {end_idx-1}")

stationary_scans = [
    data.get_velo(i)
    for i in range(start_idx, end_idx)
]

stationary_oxts = data.oxts[start_idx:end_idx]

lidar_stat = extract_all_lidar_proxies(
    stationary_scans,
    stationary_oxts
)

res = lidar_stat['icp_residual']

print("\n=== STATIONARY LIDAR ===")
print(f"Mean residual: {np.mean(res):.4f} m")
print(f"Std residual : {np.std(res):.4f} m")
print(f"Max residual : {np.max(res):.4f} m")
print(f"Min residual : {np.min(res):.4f} m")

plt.figure(figsize=(10,4))
plt.plot(res)
plt.title("Stationary LiDAR ICP Residual")
plt.xlabel("Frame Pair")
plt.ylabel("Residual (m)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ------------------------------------------------------------
# Validation Summary
# ------------------------------------------------------------

print("\n=== VALIDATION SUMMARY ===")

moving_mean = np.mean(residuals)
stationary_mean = np.mean(res)

print(f"Moving residual mean     : {moving_mean:.4f} m")
print(f"Stationary residual mean : {stationary_mean:.4f} m")

if stationary_mean < moving_mean:
    print("\nPASS: Stationary residual is lower than moving residual.")
    print("LiDAR proxy successfully captures scene motion.")
else:
    print("\nWARNING: Stationary residual is not lower than moving residual.")
    print("Further investigation may be required.")

In [ ]:
from src.proxies.gps_proxy import get_heading_gate_mask

# Trim to same length
min_len = min(len(gps['speed']), len(gps['heading_rate']), len(imu['yaw_rate']))
gate = get_heading_gate_mask(gps['speed'][:min_len], min_speed=2.0)

# Plot heading rate with gate overlay
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))

ax1.plot(gps['heading_rate'][:min_len], 'orange', alpha=0.5, label='GPS Heading Rate (raw)')
ax1.plot(imu['yaw_rate'][:min_len], 'blue', alpha=0.7, label='IMU Yaw Rate')
ax1.set_ylabel('Rate (rad/s)')
ax1.set_title('Before Gating')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gated version
gated_hr = np.where(gate, gps['heading_rate'][:min_len], np.nan)
ax2.plot(gated_hr, 'orange', alpha=0.5, label='GPS Heading Rate (gated)')
ax2.plot(imu['yaw_rate'][:min_len], 'blue', alpha=0.7, label='IMU Yaw Rate')
ax2.set_ylabel('Rate (rad/s)')
ax2.set_xlabel('Frame')
ax2.set_title('After Gating (speed < 2 m/s → heading ignored)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Stats
active = np.sum(gate)
total = len(gate)
print(f"Heading gate active: {active}/{total} frames ({active/total*100:.1f}%)")
print(f"Heading gate inactive (stationary/slow): {total - active} frames")

# Compare heading rate only where gate is active
valid = gate & ~np.isnan(gps['heading_rate'][:min_len]) & ~np.isnan(imu['yaw_rate'][:min_len])
hr_diff = np.abs(gps['heading_rate'][:min_len][valid] - imu['yaw_rate'][:min_len][valid])
if len(hr_diff) > 0:
    print(f"\nGated heading rate agreement:")
    print(f"  Mean abs diff: {np.mean(hr_diff):.4f} rad/s")
    print(f"  Max abs diff:  {np.max(hr_diff):.4f} rad/s")
else:
    print("\nNo frames passed the gate, car never exceeded 2 m/s")

In [ ]:
print("=== PROXY STATISTICS ===\n")

proxies = {
    'GPS Speed (m/s)': gps['speed'],
    'GPS Delta-V (m/s)': gps['delta_v'],
    'GPS Heading Rate (rad/s)': gps['heading_rate'],
    'IMU Delta-V (m/s)': imu['delta_v'],
    'IMU Yaw Rate (rad/s)': imu['yaw_rate'],
    'Camera Flow (px/frame)': cam['flow_magnitude'],
    'LiDAR ICP Residual (m)': lidar['icp_residual'],
}

for name, data in proxies.items():
    valid = data[~np.isnan(data)]
    print(f"{name}:")
    print(f"  mean={np.mean(valid):.4f}, std={np.std(valid):.4f}")
    print(f"  min={np.min(valid):.4f}, max={np.max(valid):.4f}")
    print()

In [ ]:
from src.features.normalization import MotionNormalizer

# Initialize normalizer
normalizer = MotionNormalizer()

# Fit on clean data
normalizer.fit(gps, imu, cam, lidar)

# Transform
z = normalizer.transform(gps, imu, cam, lidar)

# Verify: all z-scored proxies should have mean≈0, std≈1
print("\n=== VERIFICATION: Z-Scored Proxies ===")
for name, data in z.items():
    valid = data[~np.isnan(data)]
    print(f"{name:20s}: mean={np.mean(valid):+.4f}, std={np.std(valid):.4f}")

# Save for later use
normalizer.save('../src/models/normalization.pkl')